# 02. Tải dữ liệu PhysioNet 2012

Sổ tay này chỉ làm một việc: đặt hai tệp nén gốc `set-a.tar.gz` và `set-b.tar.gz` vào thư mục `Data` trên Google Drive, rồi kiểm tra tính toàn vẹn của chúng.

Vì sao tách riêng: thư mục chứa 8000 tệp `.txt` nhỏ trên Drive bị trình gắn đĩa liệt kê thiếu, làm các lần chạy sau đọc hụt hồ sơ. Giữ hai tệp nén là đủ, mọi sổ tay `W2_01_` sẽ tự giải nén ra đĩa cục bộ của phiên Colab.

Chạy một lần duy nhất. Những lần sau sổ tay tự bỏ qua nếu tệp đã đủ.

> **Vị trí trong chuỗi tái lập:** notebook 02 trên tổng số 31. Thứ tự chạy và sơ đồ phụ thuộc nằm trong `README.md`.
>
> **Cần chạy trước:** 01-Chuan_bi_thu_muc_va_ma_nguon_doi_chieu.
>
> **Thư mục kết quả:** theo mô tả trong notebook.
>
> **Mã định danh nội bộ:** `W2_00b`. Mã này còn xuất hiện trong tên khóa JSON, tên hình và nhật ký chạy; nó được giữ nguyên để không phá phép đối chiếu chéo và mã băm của bản khai báo trước.

In [1]:
#@title Tham số { display-mode: "form" }
PROJECT_DIR = "/content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4"  #@param {type:"string"}
DON_DEP_TEP_LE = True  #@param {type:"boolean"}

import os, sys, shutil, tarfile, hashlib, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
else:
    PROJECT_DIR = os.environ.get("P12_PROJECT", "./AIO2026_Conquer_M4_DLinear_NLinear_Slot4")

PROJECT = Path(PROJECT_DIR)
DATA_RAW = PROJECT / "Data"
DATA_RAW.mkdir(parents=True, exist_ok=True)
print("Thư mục dự án:", PROJECT)
print("Dữ liệu thô  :", DATA_RAW)

Mounted at /content/drive
Thư mục dự án: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4
Dữ liệu thô  : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Data


## 1. Tải hai tệp nén về thư mục Data

In [2]:
BASE = "https://physionet.org/files/challenge-2012/1.0.0"
KICH_THUOC_TOI_THIEU = 1000000


def tai_neu_thieu(part):
    tgz = DATA_RAW / (part + ".tar.gz")
    if tgz.exists() and tgz.stat().st_size > KICH_THUOC_TOI_THIEU:
        print("Đã có", tgz.name, round(tgz.stat().st_size / 1048576.0, 1), "MB")
        return tgz
    print("Đang tải", part)
    urllib.request.urlretrieve(BASE + "/" + part + ".tar.gz", tgz)
    print("Đã tải", tgz.name, round(tgz.stat().st_size / 1048576.0, 1), "MB")
    return tgz


TEP_NEN = [tai_neu_thieu("set-a"), tai_neu_thieu("set-b")]

Đang tải set-a
Đã tải set-a.tar.gz 6.3 MB
Đang tải set-b
Đã tải set-b.tar.gz 6.3 MB


## 2. Kiểm tra tính toàn vẹn

Ô dưới đây mở tệp nén và đếm số hồ sơ bên trong, không ghi tệp nhỏ nào ra Drive.

In [3]:
KIEM_KE = {}
for tgz in TEP_NEN:
    part = tgz.name.replace(".tar.gz", "")
    try:
        with tarfile.open(tgz) as tf:
            ten = [m.name for m in tf.getmembers() if m.name.endswith(".txt")]
    except tarfile.ReadError:
        raise AssertionError("Tệp nén hỏng: " + str(tgz) + ". Hãy xóa tệp này rồi chạy lại ô trên.")
    bam = hashlib.md5(tgz.read_bytes()).hexdigest()
    KIEM_KE[part] = {"so_ho_so": len(ten), "bytes": tgz.stat().st_size, "md5": bam}
    print(part, ":", len(ten), "hồ sơ .txt, md5", bam[:10])
    assert len(ten) >= 3900, "Tệp nén " + part + " thiếu hồ sơ, hãy xóa và tải lại"

print("Tổng số hồ sơ trong hai tệp nén:", sum(v["so_ho_so"] for v in KIEM_KE.values()))

set-a : 4000 hồ sơ .txt, md5 9b2e6ff663
set-b : 4000 hồ sơ .txt, md5 3b8e3a0919
Tổng số hồ sơ trong hai tệp nén: 8000


## 3. Dọn các thư mục tệp lẻ cũ trên Drive

Nếu lần chạy trước đã giải nén thẳng vào Drive, ô này xóa hai thư mục đó. Chúng vừa chiếm chỗ, vừa là nguồn của lỗi đọc hụt hồ sơ. Đặt `DON_DEP_TEP_LE` thành False nếu muốn giữ lại.

In [4]:
if DON_DEP_TEP_LE:
    for part in ("set-a", "set-b"):
        d = DATA_RAW / part
        if d.exists():
            n = len(list(d.glob("*.txt")))
            shutil.rmtree(d)
            print("Đã xóa thư mục", d.name, "chứa", n, "tệp lẻ")
        else:
            print("Không có thư mục tệp lẻ", part, "trên Drive")
else:
    print("Bỏ qua bước dọn dẹp theo lựa chọn của người dùng")

print()
print("Nội dung thư mục Data:")
for p in sorted(DATA_RAW.iterdir()):
    loai = "thư mục" if p.is_dir() else "tệp"
    print("  ", loai, p.name)
print()
print("Xong. Bước tiếp theo: chạy notebook 03-Tien_xu_ly_va_thuc_nghiem_chinh.ipynb")

Không có thư mục tệp lẻ set-a trên Drive
Không có thư mục tệp lẻ set-b trên Drive

Nội dung thư mục Data:
   tệp set-a.tar.gz
   tệp set-b.tar.gz

Xong. Bước tiếp theo: chạy notebook 03-Tien_xu_ly_va_thuc_nghiem_chinh.ipynb
